In [ ]:
# High-precision WWI medals dataset + column-filtered export from Europeana

import requests
import pandas as pd
import time
import os
import re

# =============================
# CONFIGURATION
ROWS_PER_PAGE = 100
OUTPUT_FOLDER = r"C:\Users\Admin\openscholarship\practicedata"

FULL_DATASET_FILENAME = "wwi_medals_high_precision.csv"
FILTERED_DATASET_FILENAME = "wwi_medals_filtered_columns.csv"
SAMPLE_FILENAME = "wwi_medals_sample_100.csv"
# =============================

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# -----------------------------
# Step 1: Broad Europeana query
# -----------------------------
print("Searching Europeana (broad query)...\n")

base_url = "https://api.europeana.eu/record/v2/search.json"

query = (
    '(medal OR medals) AND ('
    '"World War I" OR "First World War" OR "1914-1918" OR '
    '"Erster Weltkrieg" OR "Première Guerre mondiale"'
    ')'
)

params = {
    "query": query,
    "wskey": "apidemo",
    "rows": ROWS_PER_PAGE,
    "cursor": "*",
    "profile": "rich",
    "media": "true"
}

all_items = []
page = 1

while True:
    print(f"Fetching page {page}…")
    response = requests.get(base_url, params=params)

    if response.status_code != 200:
        print("Error:", response.status_code, response.text)
        break

    data = response.json()
    items = data.get("items", [])
    all_items.extend(items)

    next_cursor = data.get("nextCursor")
    if not next_cursor:
        print("No more pages.")
        break

    params["cursor"] = next_cursor
    page += 1
    time.sleep(0.2)

print("\nTotal items retrieved:", len(all_items))

# -----------------------------
# Step 2: Normalize dataset
# -----------------------------
df = pd.json_normalize(all_items)
print("\nDataset shape (raw):", df.shape)

# -----------------------------
# Step 3: Build combined text
# -----------------------------
text_fields = [col for col in df.columns if any(k in col.lower() for k in ["title", "subject", "type", "description"])]

df["combined_text"] = (
    df[text_fields]
    .astype(str)
    .agg(" ".join, axis=1)
    .str.lower()
)

# -----------------------------
# Step 4: HIGH-PRECISION FILTER
# -----------------------------
df_medals = df[
    df["combined_text"].str.contains(r"\bmedal(s)?\b", regex=True, na=False)
]

EXCLUDE_TERMS = [
    "coin", "banknote", "stamp",
    "poster", "photograph", "painting",
    "drawing", "print", "map"
]

exclude_pattern = "|".join(EXCLUDE_TERMS)

df_medals = df_medals[
    ~df_medals["combined_text"].str.contains(exclude_pattern, regex=True, na=False)
]

if "edmPreview" in df_medals.columns:
    df_medals = df_medals[df_medals["edmPreview"].notna()]

print("\nDataset shape (filtered, high precision):", df_medals.shape)

# -----------------------------
# Step 5: Save full dataset
# -----------------------------
full_output_path = os.path.join(OUTPUT_FOLDER, FULL_DATASET_FILENAME)
df_medals.to_csv(full_output_path, index=False)
print("\nSaved full dataset to:", full_output_path)

# -----------------------------
# Step 6: Save filtered columns dataset
# -----------------------------
SELECTED_COLUMNS = [
    "dcSubjectLangAware.en",
    "dctermsSpatial",
    "edmTimespanLabelLangAware.en",
    "edmConceptPrefLabelLangAware.en",
    "year",
    "type",
    "title",
    "provider",
    "dataProvider",
    "country",
    "completeness"
]

# Keep only existing columns (important for Europeana data)
available_columns = [col for col in SELECTED_COLUMNS if col in df_medals.columns]

df_filtered_columns = df_medals[available_columns].copy()

filtered_output_path = os.path.join(OUTPUT_FOLDER, FILTERED_DATASET_FILENAME)
df_filtered_columns.to_csv(filtered_output_path, index=False)

print("\nSaved filtered column dataset to:", filtered_output_path)
print("Columns included:", available_columns)

# -----------------------------
# Step 7: Save 100-item sample
# -----------------------------
sample_size = min(100, len(df_medals))
df_sample = df_medals.sample(sample_size, random_state=42)

sample_output_path = os.path.join(OUTPUT_FOLDER, SAMPLE_FILENAME)
df_sample.to_csv(sample_output_path, index=False)

print("Saved 100-item sample to:", sample_output_path)

# -----------------------------
# Step 8: Quick inspection
# -----------------------------
print("\nSample titles:")
print(df_sample["title"].head(10))

The current plan is to look further into ways of cleaning and better querying the data within the limitations and challenges of working with data gathered from Europeana, which is not uniform.